In [1]:
# Montage du Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# 2. Définition des chemins globaux
DRIVE_ROOT = '/content/drive/MyDrive'
REPO_DIR = f'{DRIVE_ROOT}/SpikeYOLO'
GEN1_DIR = f'{REPO_DIR}/SpikeYOLO_for_Gen1'
DATA_DIR = f'{DRIVE_ROOT}/data_p6'

# 3. Définition des fichiers de données (basés sur votre dossier data_p6)
# Fichiers d'entraînement (les deux séquences moorea)
TRAIN_FILES = [
    (f'{DATA_DIR}/moorea_2019-02-19_004_td_610500000_670500000_td.h5',
     f'{DATA_DIR}/moorea_2019-02-19_004_td_610500000_670500000_bbox.npy'),
    (f'{DATA_DIR}/moorea_2019-02-21_001_td_488500000_548500000_td.h5',
     f'{DATA_DIR}/moorea_2019-02-21_001_td_488500000_548500000_bbox.npy')
]

# Fichiers de validation
VAL_TD = f'{DATA_DIR}/validation.h5'
VAL_BBOX = f'{DATA_DIR}/validation.npy'

# Fichiers de test
TEST_TD = f'{DATA_DIR}/test.h5'
TEST_BBOX = f'{DATA_DIR}/test.npy'

# Dossiers et configuration
OUT_DIR = f'{DATA_DIR}/data_gen1_coco'
CHECKPOINTS_DIR = f'{DATA_DIR}/checkpoints'
YAML_FILE = f'{DATA_DIR}/gen4.yaml'

# 4. Déplacement dans votre dossier de travail et installation
%cd "{GEN1_DIR}"
!pip install -r ../requirements.txt
!pip install torchinfo spikingjelly

[Errno 2] No such file or directory: '/content/drive/MyDrive/SpikeYOLO/SpikeYOLO_for_Gen1'
/content
ERROR: Could not open requirements file: [Errno 2] No such file or directory: '../requirements.txt'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 14.2 MB/s eta 0:00:00


In [3]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.7 MB/s eta 0:00:00


In [8]:
import os, h5py, cv2, yaml
import numpy as np
from tqdm import tqdm

# Chemins
TRAIN_TD   = "/content/drive/My Drive/data_p6/moorea_2019-02-19_004_td_610500000_670500000_td.h5"
TRAIN_BBOX = "/content/drive/My Drive/data_p6/moorea_2019-02-19_004_td_610500000_670500000_bbox.npy"
VAL_TD     = "/content/drive/My Drive/data_p6/test.h5"
VAL_BBOX   = "/content/drive/My Drive/data_p6/test.npy"
OUT_DIR    = "/content/drive/My Drive/data_p6/data_gen1_coco"
T, OUT_H, OUT_W = 4, 320, 320

# Classes et Remapping
remap = {2: 0, 3: 1, 4: 2, 5: 3, 6: 4}
names = {0: 'car', 1: 'truck', 2: 'bus', 3: 'traffic_sign', 4: 'traffic_light'}

def timesurface_to_rgb(ts_frame):
    H, W = ts_frame.shape[1], ts_frame.shape[2]
    img = np.full((H, W, 3), 127, dtype=np.uint8)
    pos, neg = ts_frame[0], ts_frame[1]
    def norm(x):
        mx = x.max()
        return np.zeros_like(x, dtype=np.uint8) if mx == 0 else (x.astype(np.float32)/mx*255).astype(np.uint8)
    pos_n, neg_n = norm(pos), norm(neg)
    img[pos > 0, 0] = pos_n[pos > 0]
    img[pos > 0, 1] = 0; img[pos > 0, 2] = 0
    img[neg > 0, 0] = 0; img[neg > 0, 1] = 0
    img[neg > 0, 2] = neg_n[neg > 0]
    return img

def process_split(events_h5, bbox_npy, split_name, name):
    out_split = os.path.join(OUT_DIR, split_name)
    os.makedirs(out_split, exist_ok=True)
    txt_path = os.path.join(OUT_DIR, f"{split_name}.txt")

    bbox = np.load(bbox_npy, allow_pickle=True)
    if "ts" in bbox.dtype.names:
        bbox.dtype.names = ["t" if n == "ts" else n for n in bbox.dtype.names]

    unique_ts, first_idx = np.unique(bbox["t"], return_index=True)
    groups = np.split(bbox, first_idx[1:])
    ts_min, ts_max = bbox["t"].min(), bbox["t"].max()

    with h5py.File(events_h5, "r") as f:
        N = f["events"].shape[0]
        H_orig, W_orig = f["events"].shape[2], f["events"].shape[3]

        with open(txt_path, "w") as list_f:
            for idx, group in enumerate(tqdm(groups, desc=split_name)):
                ts_val = group["t"][0]
                fc = int((ts_val - ts_min) / (ts_max - ts_min) * (N - 1)) if ts_max != ts_min else N // 2
                start = max(0, min(fc - T//2, N - T))
                end   = start + T

                frames = f["events"][start:end]
                if frames.shape[0] < T:
                    pad = np.zeros((T - frames.shape[0], 2, H_orig, W_orig), dtype=frames.dtype)
                    frames = np.concatenate([pad, frames], axis=0)

                img_out = np.full((T, OUT_H, OUT_W, 3), 127, dtype=np.uint8)
                for t in range(T):
                    img_out[t] = cv2.resize(timesurface_to_rgb(frames[t]), (OUT_W, OUT_H))

                # Reshape to combine T and C dimensions: (T, H, W, C) -> (H, W, T*C)
                reshaped_img_out = img_out.reshape(OUT_H, OUT_W, T * 3)

                labels = []
                for box in group:
                    orig_cls = int(box["class_id"])
                    if orig_cls not in remap: continue
                    cx = np.clip((float(box["x"]) + float(box["w"])/2) / W_orig, 0, 1)
                    cy = np.clip((float(box["y"]) + float(box["h"])/2) / H_orig, 0, 1)
                    nw = np.clip(float(box["w"]) / W_orig, 0, 1)
                    nh = np.clip(float(box["h"]) / H_orig, 0, 1)
                    if nw > 0 and nh > 0:
                        labels.append([remap[orig_cls], cx, cy, nw, nh])

                if not labels: continue

                img_path = os.path.join(out_split, f"img_{name}_{idx:06d}.npy")
                lbl_path = os.path.join(out_split, f"img_{name}_{idx:06d}.txt")
                np.save(img_path, reshaped_img_out) # Save the reshaped array
                np.savetxt(lbl_path, np.array(labels, dtype=np.float32), fmt="%.6f")
                list_f.write(img_path + "\n")

# --- DATA GENERATION (UNCOMMENT TO RUN) ---
os.makedirs(OUT_DIR, exist_ok=True)
print("Generating training data...")
process_split(TRAIN_TD, TRAIN_BBOX, "train", "moorea_train")
print("Generating validation data...")
process_split(VAL_TD,   VAL_BBOX,   "val",   "moorea_val")
print("Data generation complete.")

# Création du fichier de configuration YAML
yaml_path = os.path.join(OUT_DIR, "gen1_moorea.yaml")
with open(yaml_path, "w") as f:
    yaml.dump({
        "path": OUT_DIR,
        "train": "train.txt",
        "val": "val.txt",
        "nc": len(names),
        "names": names
    }, f, default_flow_style=False)

print(f"✅ Configuration YAML générée : {yaml_path}")

Generating training data...


train:   4%|▍         | 132/3259 [01:31<36:03,  1.45it/s]
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

--- Logging error ---


RuntimeError: DataLoader worker (pid 10996) exited unexpectedly with exit code 1. Details are lost due to multiprocessing. Rerunning with num_workers=0 may give better error trace.